# Mendeley Example Run — Discovery + Classification

**End-to-end walkthrough** on 150 Mendeley support interactions.

This notebook demonstrates the full pipeline lifecycle:

1. **Build** — generate module files from the Excel definition
2. **Discovery** — extract a starter taxonomy from 80 sample transcripts
3. **Review & Promote** — inspect the discovered taxonomy and promote drafts to live
4. **Classification** — classify all 150 records using the discovered taxonomy
5. **Quality dashboard** — assess classification quality
6. **Per-record review** — inspect lowest-confidence classifications

**Prerequisites:**
- Python 3.9+ with: `pip install pandas openpyxl litellm tenacity python-dotenv nest-asyncio`
- A `.env` file with Azure OpenAI or OpenAI credentials
- Input file: `data/Mendeley_processed_conversations.csv`

**Estimated runtime:** ~5–10 minutes for 150 records with 2 modules.

### Prerequisites — install dependencies

In [ ]:
WORKSPACE = "mendeley"  # folder under use_cases/ to work in — change to run on another use case
import sys, os
from pathlib import Path

# Ensure project root is on sys.path and is the working directory
_root = Path().absolute()
# Navigate up until we find the rca_pipeline package
while not (_root / "use_cases").is_dir() and _root.parent != _root:
    _root = _root.parent
_root = _root / "use_cases" / WORKSPACE  # the use-case folder this notebook works in
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f"Working directory: {os.getcwd()}")


In [ ]:
# Install required dependencies (run once)
import subprocess, sys

_deps = ["pandas", "openpyxl", "litellm", "tenacity", "python-dotenv", "nest-asyncio"]
for pkg in _deps:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("All dependencies installed:")
for pkg in _deps:
    mod = __import__(pkg.replace("-", "_"))
    ver = getattr(mod, "__version__", "ok")
    print(f"  {pkg:20s} {ver}")

## 0 · Imports and setup

In [ ]:
import subprocess, sys, json, time, os
from pathlib import Path
import pandas as pd
import nest_asyncio
nest_asyncio.apply()

try:
    from rca_pipeline.config import RunConfig, get_credentials
    print("\u2713 config.py loaded")
except ImportError as e:
    print(f"\u2717 config.py: {e}")

print(f"Python {sys.version.split()[0]}")
print(f"Working directory: {os.getcwd()}")

## 1 · Configuration

In [ ]:
# ── Paths ─────────────────────────────────────────────────────
DEFINITION_FILE = "mendeley_pipeline_definition.xlsx"
DATA_FILE       = "data/Mendeley_processed_conversations.csv"
TAXONOMY_DIR    = "taxonomies"
OUTPUT_FILE     = "runs/output/Mendeley_enriched_150.xlsx"

# ── Column names (from your CSV) ─────────────────────────────
ID_COL   = "Reference #"
TEXT_COL  = "conversation"

# ── LLM settings ─────────────────────────────────────────────
MODEL       = "azure/dsaic-llm-general"
BATCH_SIZE  = 8
CONCURRENCY = 3
RETRIES     = 4
MAX_TOKENS  = 4096

# ── Discovery settings ───────────────────────────────────────
DISCOVERY_SAMPLE = 80
DISCOVERY_MODULES = ["contact_reason", "root_cause"]

# ── Enhancement toggles ──────────────────────────────────────
ENABLE_CONFIDENCE = True
TRACK_COSTS       = True
RUN_QUALITY_CHECK = True

# ── Verify input data ────────────────────────────────────────
df_preview = pd.read_csv(DATA_FILE)
print(f"Input data: {len(df_preview):,} records")
print(f"Columns: {list(df_preview.columns)}")
print(f"ID column '{ID_COL}': {df_preview[ID_COL].nunique()} unique values")
print(f"Text column '{TEXT_COL}': avg {df_preview[TEXT_COL].str.len().mean():.0f} chars")
df_preview.head(3)

## 2 · Build the pipeline

Reads the Excel definition and generates taxonomy JSONs, module `.py` files,
`pipeline_config.json`, and `build_manifest.json`.

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "rca_pipeline.cli", "build",
     "--input",        DEFINITION_FILE,
     "--taxonomy-dir", TAXONOMY_DIR,
    ],
    capture_output=True, text=True, encoding="utf-8", errors="replace"
)
print(result.stdout[-2000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])
    raise RuntimeError("rca build failed")
print("\n\u2713 Build complete")

In [ ]:
# Verify generated files
print("Generated files:")
cfg_data = json.load(open("pipeline_config.json")) if Path("pipeline_config.json").exists() else {"modules": {}}
for mod in cfg_data.get("modules", {}):
    py = Path(f"{mod}.py")
    status = f"\u2713  ({py.stat().st_size:,} bytes)" if py.exists() else "\u2717  MISSING"
    print(f"  {mod:25s}  {status}")
print()
print("Taxonomy JSONs:")
for mod, info in cfg_data.get("modules", {}).items():
    tf = info.get("taxonomy_file", "")
    if not tf: continue
    p = Path(TAXONOMY_DIR) / tf
    if p.exists():
        tax = json.load(open(p))
        n_l1 = len(tax)
        n_lf = sum(len(v.get("children",{})) for v in tax.values())
        print(f"  {tf:35s}  {n_l1} L1, {n_lf} leaf")
    else:
        print(f"  {tf:35s}  (not yet created -- will be discovered)")

## 3 · Discovery phase

Since this is a cold-start (no existing taxonomy), we sample 80 transcripts
and let the LLM extract a starter taxonomy. This runs two passes:

- **Pass 1 (extraction):** Parallel LLM batches extract abstract category phrases
- **Pass 2 (normalisation):** Single LLM call organises top items into a 2-level taxonomy

Drafts are written to `taxonomies/drafts/` for human review.

In [ ]:
print("=" * 60)
print("DISCOVERY PHASE")
print(f"  Sampling {DISCOVERY_SAMPLE} transcripts for taxonomy extraction")
print(f"  Modules: {DISCOVERY_MODULES}")
print("=" * 60)
print()

result = subprocess.run(
    [sys.executable, "-m", "rca_pipeline.cli", "discover",
     "--input",        DEFINITION_FILE,
     "--data",         DATA_FILE,
     "--taxonomy-dir", TAXONOMY_DIR,
     "--sample-n",     str(DISCOVERY_SAMPLE),
     "--top-n",        "60",
     "--batch-size",   str(BATCH_SIZE),
     "--model",        MODEL,
     "--concurrency",  str(CONCURRENCY),
     "--retries",      str(RETRIES),
     "--id-col",       ID_COL,
     "--text-col",     TEXT_COL,
     "--modules",      ] + DISCOVERY_MODULES,
    capture_output=True, text=True, encoding="utf-8", errors="replace"
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

## 4 · Review discovered taxonomy

Inspect the draft taxonomy files before promoting them to live.
This is where you would merge duplicates, rename labels, and add descriptions.

In [ ]:
draft_dir = Path(TAXONOMY_DIR) / "drafts"

if draft_dir.exists():
    drafts = sorted(draft_dir.glob("*_draft.json"))
    for d in drafts:
        tax = json.load(open(d))
        n_l1 = len(tax)
        n_lf = sum(len(v.get("children",{})) for v in tax.values())
        sep = "=" * 60
        print(f"\n{sep}")
        print(f"  {d.name}  ---  {n_l1} L1 groups, {n_lf} leaf codes")
        print(sep)
        for l1c, l1n in tax.items():
            lab = l1n.get("label", "")
            print(f"\n  {l1c}: {lab}")
            for lc, ln in l1n.get("children", {}).items():
                label = ln.get("label","") if isinstance(ln, dict) else str(ln)
                print(f"    {lc}: {label}")
else:
    print("No drafts directory found. Discovery may not have run yet.")
    print("If you already have taxonomy files in taxonomies/, skip to Section 6.")

## 5 · Promote drafts to live

After reviewing (and optionally editing) the drafts, promote them
to the live taxonomy directory so classification can use them.

In [ ]:
PROMOTE = True  # set to True after you've reviewed the drafts

if PROMOTE:
    result = subprocess.run(
        [sys.executable, "-m", "rca_pipeline.cli", "promote-drafts",
         "--input",        DEFINITION_FILE,
         "--taxonomy-dir", TAXONOMY_DIR,
        ],
        capture_output=True, text=True, encoding="utf-8", errors="replace"
    )
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)

    # Rebuild to pick up the promoted taxonomy
    result2 = subprocess.run(
        [sys.executable, "-m", "rca_pipeline.cli", "build",
         "--input", DEFINITION_FILE, "--taxonomy-dir", TAXONOMY_DIR, "--force-all"],
        capture_output=True, text=True, encoding="utf-8", errors="replace"
    )
    print("\nRebuild after promotion:")
    print(result2.stdout[-1000:])
else:
    print("Set PROMOTE = True after reviewing the drafts in Section 4.")

## 6 · Classification

Classify all 150 records using the discovered taxonomy.
This section uses the pipeline orchestrator directly for maximum control.

In [ ]:
import asyncio, logging, importlib

logging.basicConfig(level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(name)-20s | %(message)s",
    datefmt="%H:%M:%S")
logger = logging.getLogger("example_run")

# Reload all pipeline modules to pick up rebuilt .py files
for mod_name in ["config", "taxonomy", "utils", "llm_client", "pipeline",
                 "confidence", "feedback", "cost_tracker", "quality_dashboard"]:
    if mod_name in sys.modules:
        importlib.reload(sys.modules[mod_name])

from rca_pipeline.config import RunConfig, get_credentials
from rca_pipeline.taxonomy import load_taxonomies
from rca_pipeline.utils import read_input_file, drop_enrichment_cols, post_process, write_output_file
from rca_pipeline.pipeline import prepare_batched_records, run_classification_module, MODULE_REGISTRY
from rca_pipeline.llm_client import (
    set_system_prompt_suffix, clear_system_prompt_suffix,
    register_cost_tracker, set_cost_tracker_module, unregister_cost_tracker,
)
from rca_pipeline.enhancements.confidence import enrich_with_confidence, flag_low_confidence, export_review_queue, inject_confidence_instruction
from rca_pipeline.enhancements.cost_tracker import CostTracker
from rca_pipeline.enhancements.quality_dashboard import QualityDashboard

print(f"Registered modules: {list(MODULE_REGISTRY.keys())}")

creds = get_credentials()
missing = [k for k,v in creds.items() if not v]
if missing:
    print(f"\u26a0  Missing credentials: {missing}")
else:
    print("\u2713 All credentials present")

In [ ]:
# Build RunConfig
MODULES = ["contact_reason", "root_cause"]

cfg = RunConfig(
    input_file=DATA_FILE,
    output_file=OUTPUT_FILE,
    taxonomy_dir=TAXONOMY_DIR,
    id_col=ID_COL,
    text_col=TEXT_COL,
    mode="classify",
    modules=MODULES,
    batch_size=BATCH_SIZE,
    max_chars_per_batch=50000,
    model=MODEL,
    temperature=0.0,
    max_tokens=MAX_TOKENS,
    timeout=120,
    concurrency=CONCURRENCY,
    retries=RETRIES,
    enable_confidence=ENABLE_CONFIDENCE,
    confidence_threshold=0.70,
    track_costs=TRACK_COSTS,
    run_quality_check=RUN_QUALITY_CHECK,
)

print(f"Model:       {cfg.model}")
print(f"Modules:     {cfg.modules}")
print(f"Batch size:  {cfg.batch_size}")
print(f"Concurrency: {cfg.concurrency}")
print(f"Confidence:  {cfg.enable_confidence} (threshold {cfg.confidence_threshold})")
print(f"Cost track:  {cfg.track_costs}")

### Load data and taxonomies

In [ ]:
taxonomies = load_taxonomies(cfg.taxonomy_dir)
loaded = [k for k,v in taxonomies.items() if v]
empty  = [k for k,v in taxonomies.items() if not v]
print(f"Taxonomies loaded: {loaded}")
if empty:
    print(f"Empty/missing: {empty}")

df = read_input_file(cfg.input_file)
df = drop_enrichment_cols(df)
print(f"\nLoaded {len(df):,} records")
print(f"Columns: {list(df.columns)}")

batches = prepare_batched_records(df, cfg)
sizes = [len(b) for b in batches]
print(f"\nBatches: {len(batches)} | sizes: min={min(sizes)} max={max(sizes)} mean={sum(sizes)/len(sizes):.1f}")
print(f"Estimated API calls: {len(batches) * len(cfg.modules)}")

tracker = CostTracker(model=MODEL)
est = tracker.estimate_run_cost(len(df), modules=MODULES)
print(f"\nEstimated cost: ${est['estimated_total_usd']:.4f}")

### Run classification with progress

In [ ]:
# Register cost tracker
cost_tracker = CostTracker(model=MODEL)
register_cost_tracker(cost_tracker)

# Build confidence instruction
confidence_instr = inject_confidence_instruction("") if ENABLE_CONFIDENCE else ""

async def classify_all():
    all_results = []

    for mod_name in cfg.modules:
        sep = "=" * 60
        print(f"\n{sep}")
        print(f"  Classifying: {mod_name}")
        print(sep)

        # Set per-module prompt suffix (confidence + feedback)
        set_system_prompt_suffix(confidence_instr)
        set_cost_tracker_module(mod_name)

        t0 = time.time()
        results = await run_classification_module(mod_name, batches, taxonomies, cfg)
        elapsed = time.time() - t0

        # Enrich with confidence
        if ENABLE_CONFIDENCE:
            results = enrich_with_confidence(results, threshold=0.70)

        all_results.extend(results)
        print(f"  Done: {mod_name}: {len(results)} records in {elapsed:.1f}s")

    clear_system_prompt_suffix()
    return all_results

# Run
print(f"Starting classification of {len(df)} records across {len(cfg.modules)} modules...\n")
t_start = time.time()
all_results = await classify_all()
elapsed = time.time() - t_start

# Unregister cost tracker
unregister_cost_tracker()

print(f"\nClassification complete: {len(all_results)} results in {elapsed:.1f}s")
print(f"  Rate: {len(df)/elapsed:.1f} records/s across {len(cfg.modules)} modules")

## 7 · Merge and post-process

In [ ]:
# Merge by UID
grouped = {}
for r in all_results:
    uid = str(r.get("uid",""))
    grouped.setdefault(uid, {"uid": uid}).update(r)

merged = list(grouped.values())
print(f"Merged: {len(merged):,} unique UIDs (expected {len(df):,})")

errors = [r for r in merged if "_error" in r]
if errors:
    print(f"\n{len(errors)} records with errors:")
    for e in errors[:5]:
        print(f"  uid={e.get('uid')}: {str(e.get('_error',''))[:100]}")
else:
    print("No errors")

# Post-process
df_enriched = post_process(df, merged, taxonomies, cfg)

# Flag low-confidence
if ENABLE_CONFIDENCE and "_confidence" in df_enriched.columns:
    df_enriched = flag_low_confidence(df_enriched, threshold=0.70)

print(f"\nEnriched: {len(df_enriched):,} rows x {len(df_enriched.columns)} columns")
print("\nNew columns added:")
original_cols = set(df.columns)
for col in df_enriched.columns:
    if col not in original_cols:
        print(f"  {col}")

## 8 · Quality dashboard

In [ ]:
if RUN_QUALITY_CHECK:
    dashboard = QualityDashboard(df_enriched, taxonomies, cfg)
    report = dashboard.generate()
    dashboard.print_summary()
else:
    print("Quality dashboard disabled.")

### Coverage and code distribution

In [ ]:
print("Classification coverage:\n")
for col in [c for c in df_enriched.columns if c.endswith("_primary_code")]:
    filled = df_enriched[col].notna() & (df_enriched[col].astype(str).str.strip() != "")
    pct = filled.sum() / len(df_enriched) * 100
    bar = chr(9608) * int(pct / 2.5)
    print(f"  {col:45s} {filled.sum():5d}/{len(df_enriched)}  {pct:5.1f}%  {bar}")

print()
for col in [c for c in df_enriched.columns if c.endswith("_primary_label")]:
    print(f"\n{col}:")
    vc = df_enriched[col].value_counts().head(15)
    for label, cnt in vc.items():
        pct = cnt / len(df_enriched) * 100
        bar = chr(9608) * int(pct / 2)
        print(f"  {str(label):50s} {cnt:4d}  ({pct:4.1f}%)  {bar}")

### Confidence analysis

In [ ]:
if "_confidence" in df_enriched.columns:
    conf = pd.to_numeric(df_enriched["_confidence"], errors="coerce")
    valid = conf.dropna()

    if len(valid) > 0:
        print(f"Confidence scores: {len(valid):,} records\n")
        print(f"  Mean:   {valid.mean():.3f}")
        print(f"  Median: {valid.median():.3f}")
        print(f"  Std:    {valid.std():.3f}")
        print(f"  Min:    {valid.min():.3f}")
        print(f"  Max:    {valid.max():.3f}")

        print(f"\nDistribution:")
        for label, lo, hi in [("Very high (0.9-1.0)", 0.9, 1.01),
                               ("High (0.7-0.9)", 0.7, 0.9),
                               ("Moderate (0.5-0.7)", 0.5, 0.7),
                               ("Low (0.0-0.5)", 0.0, 0.5)]:
            n = ((valid >= lo) & (valid < hi)).sum()
            pct = n / len(valid) * 100
            print(f"  {label:25s}  {n:5d}  ({pct:5.1f}%)")

        if "_needs_review" in df_enriched.columns:
            n_review = df_enriched["_needs_review"].sum()
            print(f"\nFlagged for human review: {n_review:,} ({n_review/len(df_enriched)*100:.1f}%)")
    else:
        print("No valid confidence scores found.")
else:
    print("Confidence scoring disabled.")

## 9 · Per-record review (lowest confidence)

In [ ]:
REVIEW_N = 5

review_cols = [c for c in [
    ID_COL, TEXT_COL,
    "contact_reason_primary_label",
    "root_cause_primary_label", "root_cause_justification",
    "root_cause_evidence",
    "_confidence", "_needs_review",
] if c in df_enriched.columns]

if "_confidence" in df_enriched.columns:
    review_df = df_enriched.sort_values("_confidence", ascending=True).head(REVIEW_N)
    print(f"Reviewing {REVIEW_N} lowest-confidence records:\n")
else:
    review_df = df_enriched.sample(n=min(REVIEW_N, len(df_enriched)), random_state=42)
    print(f"Reviewing {REVIEW_N} random records:\n")

pd.set_option("display.max_colwidth", 120)
for _, row in review_df.iterrows():
    print("=" * 80)
    for col in review_cols:
        val = row.get(col, "")
        if col == TEXT_COL and isinstance(val, str) and len(val) > 200:
            val = val[:200] + " [...]"
        print(f"  {col:40s}: {val}")
print("=" * 80)

## 10 · Cost summary

In [ ]:
if TRACK_COSTS and cost_tracker is not None:
    cost_tracker.print_summary()

    Path("cost_reports").mkdir(exist_ok=True)
    cost_tracker.save(f"cost_reports/costs_{cost_tracker._run_id}.json")
else:
    print("Cost tracking disabled.")

## 11 · Write output

In [ ]:
Path("output").mkdir(exist_ok=True)
out_path = write_output_file(df_enriched, cfg.output_file)
print(f"\nOutput: {out_path}")
print(f"  {len(df_enriched):,} rows x {len(df_enriched.columns)} columns")

# Export review queue
if ENABLE_CONFIDENCE and "_needs_review" in df_enriched.columns:
    review_path = "runs/output/review_queue.xlsx"
    export_review_queue(df_enriched, review_path)

# Save quality report
if RUN_QUALITY_CHECK and 'dashboard' in dir():
    dashboard.save("runs/output/quality_report.json")
    try:
        dashboard.save_excel("runs/output/quality_report.xlsx")
    except Exception as e:
        print(f"Could not save Excel quality report: {e}")

print("\nAll output files:")
for f in sorted(Path("output").glob("*")):
    print(f"  {f.name:40s}  ({f.stat().st_size/1024:.1f} KB)")

## 12 · Run summary

In [ ]:
print("=" * 60)
print("  EXAMPLE RUN COMPLETE")
print("=" * 60)
print(f"  Input:       {DATA_FILE}")
print(f"  Records:     {len(df_enriched):,}")
print(f"  Modules:     {cfg.modules}")
print(f"  Output:      {cfg.output_file}")
print(f"  Columns:     {len(df_enriched.columns)}")
if "_confidence" in df_enriched.columns:
    conf = pd.to_numeric(df_enriched["_confidence"], errors="coerce")
    print(f"  Avg conf:    {conf.mean():.3f}")
    n_review = df_enriched.get("_needs_review", pd.Series()).sum()
    print(f"  Needs review: {n_review}")
errors = [r for r in merged if "_error" in r]
print(f"  Errors:      {len(errors)}")
print(f"  Elapsed:     {elapsed:.1f}s")
print("=" * 60)
print()
print("Next steps:")
print("  1. Open output/Mendeley_enriched_150.xlsx to review classifications")
print("  2. Open output/review_queue.xlsx for low-confidence records")
print("  3. Open output/quality_report.xlsx for the quality dashboard")
print("  4. Add corrections via FeedbackStore for the next run")